# 00 – Project Setup
## Procurement Analytics using Medallion Architecture
### Project Objective

This notebook initializes the local PySpark environment for the Procurement Analytics project. The goal is to verify the Spark setup, validate dataset availability, and prepare the project structure for implementing the Medallion Architecture (Bronze, Silver, Gold).

### Technologies Used
- Python
- PySpark
- Pandas
- Jupyter Notebook (VS Code)

## Step 1: Initialize Spark Session

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('ProcurementAnalytics') \
    .master('local[*]') \
    .getOrCreate()

print('Spark Started Successfully')
print('Spark Version:', spark.version)

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Started Successfully
Spark Version: 4.2.0


### Explanation

* `SparkSession.builder` creates a Spark session.
* `local[*]` uses all available CPU cores on the local machine.
* The session will be used for all subsequent data processing tasks.

---

# Step 2: Verify Data Files


In [3]:
import os

print(os.listdir('data'))

['contracts_1.csv', 'invoices_1.csv', 'orders_1.csv', 'vendors_1.csv']


### Explanation

This step confirms that all procurement datasets are available inside the `data` folder before processing begins.

# Step 3: Load Purchase Orders Dataset


In [4]:
po_df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('data/orders_1.csv')

po_df.printSchema()
po_df.show(5, truncate=False)

root
 |-- po_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- quantity_requested: integer (nullable = true)
 |-- po_timestamp: string (nullable = true)

+---------+---------+-------------+------------------+-------------------+
|po_id    |vendor_id|item_name    |quantity_requested|po_timestamp       |
+---------+---------+-------------+------------------+-------------------+
|PO0000001|V00637   |E-markets    |487               |2025-04-08 15:51:30|
|PO0000002|V01075   |Channels     |326               |2025-05-21 20:44:55|
|PO0000003|V03667   |Models       |24                |2024-05-20 14:26:10|
|PO0000004|V01296   |Architectures|756               |2024-10-18 02:03:33|
|PO0000005|V04012   |Bandwidth    |554               |2025-10-11 07:07:53|
+---------+---------+-------------+------------------+-------------------+
only showing top 5 rows


### Explanation

The Purchase Orders dataset is loaded into a Spark DataFrame. Schema inference automatically detects column data types, and the first five records are displayed for verification.

# Step 4: Load Invoices Dataset


In [5]:
invoice_df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('data/invoices_1.csv')

invoice_df.printSchema()
invoice_df.show(5, truncate=False)

root
 |-- invoice_id: string (nullable = true)
 |-- po_id: string (nullable = true)
 |-- invoiced_price_per_unit: string (nullable = true)
 |-- invoice_timestamp: timestamp (nullable = true)

+-----------+---------+-----------------------+-------------------+
|invoice_id |po_id    |invoiced_price_per_unit|invoice_timestamp  |
+-----------+---------+-----------------------+-------------------+
|INV00000001|PO0000665|3865.0                 |2024-08-04 04:12:27|
|INV00000002|PO0001617|282.25                 |2026-02-09 05:45:13|
|INV00000003|PO0002057|4743.39                |2026-01-04 11:32:19|
|INV00000004|PO0001872|4129.53                |2024-06-20 16:52:09|
|INV00000005|PO0003369|3080.2                 |2026-03-29 06:08:10|
+-----------+---------+-----------------------+-------------------+
only showing top 5 rows


### Explanation

The Invoices dataset is loaded into Spark and previewed to validate successful ingestion.

# Step 5: Load Vendor  Dataset


In [6]:
contract_df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('data/vendors_1.csv')

contract_df.printSchema()
contract_df.show(5, truncate=False)

root
 |-- vendor_id: string (nullable = true)
 |-- vendor_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- risk_rating: string (nullable = true)

+---------+----------------------+------+-----------+
|vendor_id|vendor_name           |region|risk_rating|
+---------+----------------------+------+-----------+
|V04649   |Dixon-Hughes          |LATAM |Medium     |
|V04163   |Hansen PLC            |EMEA  |High       |
|V03664   |Klein, Chavez and Lane|APAC  |Medium     |
|V03444   |Perez-Harmon          |APAC  |High       |
|V02551   |Taylor Inc            |LATAM |Low        |
+---------+----------------------+------+-----------+
only showing top 5 rows


# Step 6: Load Vendor Contracts Dataset


In [7]:
contracts_df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv('data/contracts_1.csv')

contracts_df.printSchema()
contracts_df.show(5, truncate=False)

root
 |-- contract_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- negotiated_price: double (nullable = true)
 |-- valid_from: string (nullable = true)

+-----------+---------+-----------+----------------+----------+
|contract_id|vendor_id|item_name  |negotiated_price|valid_from|
+-----------+---------+-----------+----------------+----------+
|C001960    |V02900   |Communities|888.12          |2025-02-24|
|C001369    |V03799   |Channels   |3717.1          |2026-03-25|
|C004832    |V02993   |Convergence|3054.3          |2024-07-30|
|C004317    |V02067   |Users      |3520.68         |2024-01-20|
|C003307    |V04410   |E-tailers  |3561.33         |2024-12-19|
+-----------+---------+-----------+----------------+----------+
only showing top 5 rows


### Explanation

The Vendor Contracts dataset is loaded and previewed to ensure that contract-related information is available for downstream SCD Type 2 processing.

# Step 7: Record Count Validation


In [8]:
print('Purchase Orders :', po_df.count())
print('Invoices        :', invoice_df.count())
print('Contracts       :', contract_df.count())

Purchase Orders : 5000
Invoices        : 5000
Contracts       : 5100


### Explanation

Record counts are validated to ensure that all datasets have been loaded correctly and no records were lost during ingestion.

# Conclusion

In this notebook, the local PySpark environment was successfully initialized, procurement datasets were verified and loaded into Spark DataFrames, and the basic project structure was validated. The setup is now ready for implementing the Bronze layer ingestion process in the next notebook.
